# Opérations ETL avec l'API des Dataframes

### Objectifs : 
- Implémenter les opérations communes d'un traitement ETL en utilisant les Dataframes Spark
- Nettoyage de la données et conversion
- Créer des caractéristiques dérivées par le biais de transformations
- Utilisations des différentes méthodes de références de colonnes
- Travailler avec des UDFs

### Préparation du notebook

Dans ce notebook il y a un paramètre `full_path` en haut du notebook

![](./Resources/NB03/img_parameters.png)

La cellule dessous permet de créer ce paramètre si il n'existe pas, il y aura une valeur par défaut vers l'emplacement du fichier netflix_titles.csv (fichier télécharger de puis Kaggle) : `/Workspace/Users/{username}/databricks-training/Spark Developer/Introdution a Spark/Resources/NB03/`

**À propos de ce jeu de données** : Netflix est l'une des plateformes de streaming vidéo et média les plus populaires. Elle propose plus de 8000 films ou séries disponibles sur sa plateforme et, à la mi-2021, compte plus de 200 millions d'abonnés dans le monde. Ce jeu de données tabulaire contient la liste de tous les films et séries disponibles sur Netflix, ainsi que des informations telles que : casting, réalisateurs, notes, année de sortie, durée, etc.

[Lien vers le Dataset Kaggle](https://www.kaggle.com/datasets/shivamb/netflix-shows)

Comme Spark ne peut pas lire de fichier directement dans le système de fichier du Workspace comme on l'as fait avec Pands, il faut : 
- Créer un volume (dans un Catalog/schema)
- Placer le fichier dans le volume

Les deux prochaines cellules vont paramètrer les différentes variables nécéssaires pour le `Setup`. 

Par défaut un volume sera créer dans le catalog : **workspace.default** (spark_training) mais on peut modifier les widgets (directement dans le code ou si la cellules à déjà été exécuté en haut du notebook).

In [0]:
# Passe la variable 'full_path' avec le chemin vers le fichier CSV MOCK_DATA.csv au notebook NB02/Setup
import os

full_path = os.getcwd() + "/Resources/NB03/"

dbutils.widgets.text("full_path", full_path)
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "default")
dbutils.widgets.text("volume", "spark_training")

In [0]:
%run ./Resources/NB03/Setup

## Chargement des données et inspéction du Dataset


In [0]:
# lecture du dataset dans un Dataframe Spark

netflix_df = spark.read.csv(full_path + "netflix_titles.csv", header=True)
display(netflix_df)

In [0]:
# affichage du schéma
netflix_df.printSchema()

## CARACTÉRISTIQUES

Le schéma du DataFrame `netflix_df` décrit la structure des données importées depuis le fichier `netflix_titles.csv`. Il contient les colonnes suivantes :

- **show_id** : Identifiant unique du contenu (String)
- **type** : Type de contenu ("Movie" ou "TV Show") (String)
- **title** : Titre du contenu (String)
- **director** : Nom du ou des réalisateurs (String)
- **cast** : Liste des acteurs principaux (String)
- **country** : Pays d’origine (String)
- **date_added** : Date d’ajout sur Netflix (String)
- **release_year** : Année de sortie (Integer)
- **rating** : Classification d’âge (String)
- **duration** : Durée (ex: "90 min" ou "1 Season") (String)
- **listed_in** : Catégories/genres (String)
- **description** : Brève description du contenu (String)

Ce schéma permet d’effectuer des opérations de nettoyage, de transformation et d’analyse sur les différents attributs des films et séries disponibles sur Netflix.

### Traitement des valeurs `null`

- Vérifier les colonnes obligatoires (show_id, type, title, director, cast, rating) et intégrer ces lignes dans un tables de catalog car ces lignes peuvent être améliorées ou corrigées plus tard

In [0]:
# récupération des lignes invalides (avec valeurs null) il devrait y avoir 3109 lignes
netflix_content_invalid = netflix_df.filter(netflix_df.type.isNull() | netflix_df.title.isNull() | netflix_df.director.isNull() | netflix_df.cast.isNull())
display(netflix_content_invalid)

In [0]:
# Ecriture des données dans la table netflix_content_invalid
netflix_content_invalid.write.saveAsTable("netflix_content_invalid")

### Définition du schéma du Dataframe

On va créer 2 schémas : 
- tv_shows
- movies

#### TV Shows
- **show_id** : Identifiant unique du contenu (String)
- **title** : Titre du contenu (String)
- **director** : Nom du ou des réalisateurs (String)
- **cast** : Liste des acteurs principaux (Array<String>)
- **country** : Pays d’origine (String)
- **date_added** : Date d’ajout sur Netflix (Date)
- **release_year** : Année de sortie (Integer)
- **rating** : Classification d’âge (String)
- **season** : Durée (ex: "1 Season") (String)
- **listed_in** : Catégories/genres (Array<String>)
- **description** : Brève description du contenu (String)


#### Movies

- **show_id** : Identifiant unique du contenu (String)
- **title** : Titre du contenu (String)
- **director** : Nom du ou des réalisateurs (String)
- **cast** : Liste des acteurs principaux (Array<String>)
- **country** : Pays d’origine (String)
- **date_added** : Date d’ajout sur Netflix (Date)
- **release_year** : Année de sortie (Integer)
- **rating** : Classification d’âge (String)
- **duration** : Durée en minute(ex: "90") (Integer)
- **listed_in** : Catégories/genres (Array<String>)
- **description** : Brève description du contenu (String)


Pour créer une colonne contenant des valeurs sous forme de liste dans un DataFrame Spark, on peut utiliser la fonction `split` pour transformer une chaîne de caractères en liste, ou directement définir une colonne de type Array. Exemple :

python
from pyspark.sql import SparkSession
from pyspark.sql.functions import split

# Exemple de DataFrame avec une colonne 'cast' sous forme de chaîne
df = spark.createDataFrame([
    ("Mayur More, Jitendra Kumar, Ranjan Raj, Alam Khan, Ahsaas Channa, Revathi Pillai, Urvi Singh, Arun Kumar",)
], ["cast"])

# Transformation de la colonne 'cast' en liste (ArrayType)
df_with_list = df.withColumn("cast_list", split(df["cast"], ",\s*"))
display(df_with_list)


La colonne `cast_list` contiendra alors une liste de noms d’acteurs.

Dans une table du catalog, une colonne de type liste (Array) sera stockée avec le type `ARRAY<STRING>`. Par exemple, après transformation, la colonne `cast_list` apparaîtra dans le schéma de la table comme :

- **cast_list** : ARRAY<STRING>

Cela permet d’effectuer des opérations SQL sur les éléments de la liste, comme des filtres ou des recherches, directement dans la table du catalog.

In [0]:
netflix_content_nonnull_df = netflix_df.na.drop(
    how='any',
    subset=['type', 'title', 'director', 'cast']
)

display(netflix_content_nonnull_df)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType

tvshows_schema = StructType([
    StructField("show_id", StringType(), True),
    StructField("title", StringType(), True),
    StructField("director", StringType(), True),
    StructField("cast", ArrayType(), True),
    StructField("country", StringType(), False),
    StructField("date_added", StringType(), False),
    StructField("release_year", IntegerType(), False),
    StructField("rating", StringType(), False),
    StructField("duration", StringType(), False),
    StructField("listed_in", ArrayType(), False),
    StructField("description", StringType(), False)
])

from pyspark.sql.functions import split

# Transformation des colonnes 'cast' et 'listed_in' en ArrayType
netflix_df_array = netflix_df.withColumn("cast", split(netflix_df["cast"], ",\s*")) \
    .withColumn("listed_in", split(netflix_df["listed_in"], ",\s*"))

display(netflix_df_array)

movies_schema = StructType([
    StructField("show_id", StringType(), True),
    StructField("title", StringType(), True),
    StructField("director", StringType(), True),
    StructField("cast", ArrayType(), True),
    StructField("country", StringType(), False),
    StructField("date_added", StringType(), False),
    StructField("release_year", IntegerType(), False),
    StructField("rating", StringType(), False),
    StructField("duration", IntegerType(), False),
    StructField("listed_in", ArrayType(), False),
    StructField("description", StringType(), False)
])